# LMS Optical / Householder Widgets

Interactive notebooks for the reflected-ray interpretation of the finite-$N$ LMS reduced force.

For frozen canonical shape constants $p_i \in S^{d-1}$, the optical drive is

$$
R_p(w)=\sum_i a_i H_{p_i-w}(p_i)=-\sum_i a_i M_w(p_i),
\qquad
\dot w=\frac12(1-|w|^2)R_p(w).
$$

The widgets below expose the hidden reflected cloud $r_i(w)$, its barycenter $R_p(w)$, ray coherence $|R_p(w)|$, ray variance $1-|R_p(w)|^2$, covariance axes, selected-anchor reflection geometry, and in 2D the composite Busemann wavefront $S_p(w)$.


In [1]:
from pathlib import Path
import sys

# This notebook lives in kuramoto/LMSSPP/notebooks.
# Add the local LMSSPP source tree without requiring package installation.
LMSSPP_SRC = Path("../src").resolve()
if not LMSSPP_SRC.exists():
    # Fallback for running the notebook from the repository root.
    LMSSPP_SRC = Path("pitch-website/public/notebooks/kuramoto/LMSSPP/src").resolve()
if str(LMSSPP_SRC) not in sys.path:
    sys.path.insert(0, str(LMSSPP_SRC))

print("LMSSPP source:", LMSSPP_SRC)


LMSSPP source: /Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src


In [2]:
import numpy as np
import torch

from lmsspp.lms_optical_widget import (
    LMSOpticalDiskWidget,
    LMSOpticalDynamicInversionCayleyDiskWidget,
    LMSOpticalDynamicInversionBall3DWidget,
    LMSOpticalWeightedCayleyDiskWidget,
    LMSOpticalHyperboloidScreenWidget,
    optical_state,
    mobius_reflection_error,
)


## Quick Identity Check

This verifies the core optical identity used by the widgets:

$$H_{p_i-w}(p_i)=-M_w(p_i).$$


In [3]:
rng = np.random.default_rng(5)
P = torch.as_tensor(rng.normal(size=(32, 3)), dtype=torch.float64)
P = P / P.norm(dim=-1, keepdim=True)
w = torch.tensor([0.23, -0.11, 0.19], dtype=torch.float64)

err = mobius_reflection_error(w, P)
state = optical_state(w, P)
print(f"Householder/Mobius identity error: {float(err):.3e}")
print(f"coherence |R|: {float(state.coherence):.6f}")
print(f"ray variance 1-|R|^2: {float(state.variance):.6f}")


Householder/Mobius identity error: 2.593e-09
coherence |R|: 0.655850
ray variance 1-|R|^2: 0.569861


## 2D Optical Disk Widget

This view is best for the wavefront/eikonal picture. It shows contours of $S_p(w)$, the current $w$, the optical drive $R(w)$, the reflected cloud on $S^1$, a vector-sum polygon, and selected-anchor Householder geometry.

Use the sliders as notebook-safe drag handles for $w$ and the selected anchor. The **Find optical balance** button moves $w$ to the canonical balance point where $R(w)\approx 0$.


In [4]:
disk = LMSOpticalDiskWidget(N=18, preset="random", seed=7, grid_size=120)
disk.layout

## 2D Optical Disk Widget with Weighted / Regular Cayley Chart

The toggle on the playback row switches between the weighted total-Busemann Cayley chart and the regular moving-puncture Cayley chart with $\xi_t=w_t/|w_t|$.


In [5]:
cayley = LMSOpticalWeightedCayleyDiskWidget(N=18, preset="random", seed=7, grid_size=80)
cayley.layout


## 2D Dynamic W-Relative Charts

The bottom-left panel reuses the first disk panel data with a one-sided unit-distance normalization around $w_t$: points inside the unit neighborhood of $w_t$ are pushed radially outward to distance $1$, while points already at distance at least $1$ are unchanged. The bottom-right panel keeps the spherical inversion around $-w_t$.


In [6]:
dynamic_inversion = LMSOpticalDynamicInversionCayleyDiskWidget(N=18, preset="random", seed=7, grid_size=80)
dynamic_inversion.layout


## 3D Dynamic Inversion Chart

A single-scene 3D version of the dynamic inversion chart. It exact-centers a cloud on $S^2$, precomputes the reduced orbit with the shared physical/ES target-radius logic, and displays $p_i^0$, $w(t)$, and $S^2$ through $(x+w_t)/|x+w_t|^2$.


In [7]:
dynamic_inversion_3d = LMSOpticalDynamicInversionBall3DWidget(N=18, preset="random", seed=7, grid_size=80)
dynamic_inversion_3d.layout
    

/opt/anaconda3/envs/manip311/lib/python3.11/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



## 2D Hyperboloid + Optical Screen Widget

This view keeps the reflected-cloud/level-set evolution panel and replaces the other panels with the hyperboloid null-data chart plus the local optical screen spread diagnostic.


In [8]:
hyperboloid = LMSOpticalHyperboloidScreenWidget(N=18, preset="random", seed=7, grid_size=80)
display(hyperboloid.layout)
